### About Dataset

Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.


Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration

- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis

### Best Practices

1) Preprocessing and cleaning
2) Train test split
3) Bow,Tf-Idf,Word2vec (any 1)
4) ML Algo training
5) Evaluation report

In [66]:
import pandas as pd
data=pd.read_csv('../datasets/all_kindle_review.csv',index_col=0)
data.head()

,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [67]:
data=data[['reviewText','rating']]
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [68]:
data.shape

(12000, 2)

In [69]:
data.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [70]:
data['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [71]:
data['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [72]:
### Preprocessing
## positive review-1, neg review=0

data['rating']=data['rating'].apply(lambda x: 0 if x<3 else 1)
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [73]:
data['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [74]:
## 1. Lower all the text

data['reviewText']=data['reviewText'].str.lower()

In [75]:
import re
from nltk.corpus import  stopwords
stopwrds=stopwords.words('english')

In [76]:
from bs4 import BeautifulSoup

In [77]:
## Remove html tags
data['reviewText']=data['reviewText'].apply(lambda x: BeautifulSoup(x,'html.parser').get_text())
## Remove the url
data['reviewText']=data['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove all special chars
data['reviewText']=data['reviewText'].apply(lambda x:re.sub('[^a-z A-Z 0-9-]+','',x))
## Remove the stopwords
data['reviewText']=data['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwrds]))
## Remove any additional spaces
data['reviewText']=data['reviewText'].apply(lambda x:" ".join(x.split()))

In [78]:
data.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [79]:
### Lemmatization

from nltk.stem import WordNetLemmatizer
lemmatize=WordNetLemmatizer()


In [80]:
def lemmatize_words(text):
    return " ".join([lemmatize.lemmatize(word) for word in text.split()])

In [81]:
data['reviewText']=data['reviewText'].apply(lambda x:lemmatize_words(x))

In [82]:
data.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


### Train test split

In [87]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(data['reviewText'],data['rating'],test_size=0.2)

In [88]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(x_train).toarray()
X_test_bow=bow.transform(x_test).toarray()

In [89]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [91]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)

In [93]:
from sklearn.metrics import accuracy_score,classification_report
y_pred=nb_model_bow.predict(X_test_bow)


In [94]:
print(accuracy_score(y_test,y_pred))

0.5816666666666667


In [95]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.41      0.67      0.51       775
           1       0.78      0.54      0.64      1625

    accuracy                           0.58      2400
   macro avg       0.59      0.61      0.57      2400
weighted avg       0.66      0.58      0.59      2400

